## Graph E2E 테스트

<div style="text-align: right"> <b>KMYU</b></div>
<div style="text-align: right"> Initial issue : 2026.03.20 </div>
<div style="text-align: right"> last update : 2026.03.20 </div>

개정 이력  
- `2026.03.20` : 노트북 최초 작성

**사전 요구사항:**
- PostgreSQL 실행 (`make docker-up && make migrate && make seed`)
- `.env` 파일에 API 키 설정 (OpenAI/Anthropic, Pinecone)
- LangGraph 그래프 전체를 End-to-End로 실행

In [ ]:
import sys

sys.path.insert(0, "../../")

from dotenv import load_dotenv

load_dotenv()

### 1. 그래프 빌드

In [ ]:
from backend.app.agent.graph import build_graph

graph = build_graph()
print(f"Graph type: {type(graph).__name__}")

In [ ]:
# 노드 및 엣지 구조 확인
graph_def = graph.get_graph()

print("=== Nodes ===")
for node in graph_def.nodes:
    print(f"  - {node}")

print("\n=== Edges ===")
for edge in graph_def.edges:
    print(f"  {edge}")

### 2. 그래프 시각화

In [ ]:
# Mermaid 다이어그램 출력
mermaid = graph_def.draw_mermaid()
print(mermaid)

In [ ]:
# Jupyter에서 Mermaid 렌더링 (IPython 지원 시)
from IPython.display import Image, display

try:
    img_bytes = graph_def.draw_mermaid_png()
    display(Image(img_bytes))
except Exception as e:
    print(f"PNG 렌더링 불가 (mermaid-cli 필요): {e}")
    print("위 Mermaid 코드를 https://mermaid.live 에 붙여넣어 확인 가능")

### 3. 정상 딜 E2E 실행

In [ ]:
import uuid

# 충분한 정보가 포함된 정상 딜 입력
normal_deal_input = """고객사: 메가테크 (대기업, 제조업)
프로젝트: 공장 설비 예지보전 AI 시스템 개발
배경: 기존 정기 점검 체계에서 AI 기반 예측 유지보수로 전환하여 설비 가동률 향상 및 유지보수 비용 절감
기술 요구사항: Python, TensorFlow, 시계열 분석, IoT 센서 데이터 처리, 대시보드 (React)
예상 규모: 2억원
기간: 5개월
결제 조건: 착수금 30%, 중간 40%, 완료 30%
특이사항: 공장 내 보안 네트워크 환경에서만 운영, 온프레미스 배포 필수"""

test_deal_id = str(uuid.uuid4())
print(f"Deal ID: {test_deal_id}")

In [ ]:
# graph.ainvoke로 전체 파이프라인 실행
result = await graph.ainvoke({
    "deal_id": test_deal_id,
    "deal_input": normal_deal_input,
})

print(f"실행 완료! Result keys: {list(result.keys())}")

### 4. 결과 분석

In [ ]:
import json

print("=== 핵심 결과 ===")
print(f"Status: {result.get('status')}")
print(f"Verdict: {result.get('verdict')}")
print(f"Total Score: {result.get('total_score')}")
print(f"Errors: {result.get('errors', [])}")

In [ ]:
# structured_deal 확인
print("=== Structured Deal ===")
print(json.dumps(result.get("structured_deal", {}), ensure_ascii=False, indent=2))

In [ ]:
# scores 확인
print("=== Scores ===")
for s in result.get("scores", []):
    print(f"  {s['criterion']}: {s['score']}점 (가중 {s['weighted_score']}) — {s['rationale']}")
print(f"\n  총점: {result.get('total_score')} → {result.get('verdict')}")

In [ ]:
# resource_estimate 확인
print("=== Resource Estimate ===")
print(json.dumps(result.get("resource_estimate", {}), ensure_ascii=False, indent=2))

In [ ]:
# risks 확인
print("=== Risks ===")
for r in result.get("risks", []):
    print(f"  [{r.get('level')}] {r.get('category')}: {r.get('item')} — {r.get('description')}")

In [ ]:
# similar_projects 확인
print("=== Similar Projects ===")
similar = result.get("similar_projects", [])
if similar:
    print(json.dumps(similar, ensure_ascii=False, indent=2))
else:
    print("유사 프로젝트 없음")

In [ ]:
# final_report 마크다운 렌더링
from IPython.display import Markdown, display

display(Markdown(result.get("final_report", "리포트 없음")))

### 5. Hold 케이스 테스트

In [ ]:
# 정보가 부족한 딜 입력 → Hold 경로 확인
hold_deal_input = "AI 프로젝트를 하고 싶습니다. 예산이나 기간은 아직 미정입니다."

hold_deal_id = str(uuid.uuid4())
print(f"Hold Deal ID: {hold_deal_id}")

In [ ]:
hold_result = await graph.ainvoke({
    "deal_id": hold_deal_id,
    "deal_input": hold_deal_input,
})

print("=== Hold 케이스 결과 ===")
print(f"Status: {hold_result.get('status')}")
print(f"Verdict: {hold_result.get('verdict')}")
print(f"Total Score: {hold_result.get('total_score')}")
print(f"Missing Fields: {hold_result.get('structured_deal', {}).get('missing_fields', [])}")
print(f"\nFinal Report:\n{hold_result.get('final_report', '')}")

### 6. 스트리밍 실행 (단계별 진행 확인)

In [ ]:
# graph.astream으로 노드별 실행 결과를 실시간 확인
stream_deal_id = str(uuid.uuid4())

print("=== 스트리밍 실행 시작 ===")
accumulated = {}

async for event in graph.astream(
    {"deal_id": stream_deal_id, "deal_input": normal_deal_input},
    stream_mode="updates",
):
    for node_name, node_output in event.items():
        print(f"\n--- [{node_name}] 완료 ---")
        print(f"  Output keys: {list(node_output.keys())}")

        # 주요 필드 요약 출력
        if "structured_deal" in node_output:
            sd = node_output["structured_deal"]
            print(f"  customer: {sd.get('customer_name')}, industry: {sd.get('customer_industry')}")
            print(f"  missing_fields: {sd.get('missing_fields', [])}")
        if "verdict" in node_output:
            print(f"  verdict: {node_output['verdict']}, score: {node_output.get('total_score')}")
        if "resource_estimate" in node_output:
            re = node_output["resource_estimate"]
            print(f"  duration: {re.get('duration_months')}개월, cost: {re.get('total_cost')}")
        if "risks" in node_output:
            print(f"  risks count: {len(node_output['risks'])}")
        if "similar_projects" in node_output:
            print(f"  similar_projects count: {len(node_output['similar_projects'])}")
        if "final_report" in node_output:
            print(f"  final_report length: {len(node_output['final_report'])} chars")

        accumulated.update(node_output)

print("\n=== 스트리밍 실행 완료 ===")

### 7. 에러 핸들링 테스트

In [ ]:
# 빈 입력으로 실행 → 에러 필드 누적 확인
error_deal_id = str(uuid.uuid4())

error_result = await graph.ainvoke({
    "deal_id": error_deal_id,
    "deal_input": "",
})

print("=== 빈 입력 결과 ===")
print(f"Status: {error_result.get('status')}")
print(f"Verdict: {error_result.get('verdict')}")
print(f"Errors: {error_result.get('errors', [])}")
print(f"Final Report:\n{error_result.get('final_report', '')}")